# ============================================================
# Cell 0: Install vLLM from bundled wheels (no internet)
# ============================================================
import os, sys, subprocess, glob

# Debug: show what's available
print("=== /kaggle/input contents ===")
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth < 3:
        indent = "  " * depth
        for d in dirs:
            print(f"{indent}{d}/")
        for f in files[:10]:  # limit file listing
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f"{indent}{f} ({size_mb:.1f} MB)")
        if len(files) > 10:
            print(f"{indent}... and {len(files)-10} more files")

print("\n=== Python version ===")
print(sys.version)

print("\n=== Installing vLLM ===")
# Extract wheels archive
archive = '/kaggle/input/aimo-3-utils/wheels.tar.gz'
temp_dir = '/kaggle/tmp/setup'

if os.path.exists(archive):
    print(f"Found archive: {archive}")
    if not os.path.exists(os.path.join(temp_dir, 'wheels')):
        os.makedirs(temp_dir, exist_ok=True)
        result = subprocess.run(['tar', '-xzf', archive, '-C', temp_dir])
        print(f"tar exit code: {result.returncode}")
    
    # Show what was extracted
    wheels_dir = os.path.join(temp_dir, 'wheels')
    if os.path.exists(wheels_dir):
        whl_files = os.listdir(wheels_dir)
        print(f"Extracted {len(whl_files)} files to {wheels_dir}")
        vllm_wheels = [w for w in whl_files if 'vllm' in w.lower()]
        print(f"vLLM wheels: {vllm_wheels}")
    
    # Install - show full output for debugging
    result = subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--no-index', '--find-links', wheels_dir,
        '--no-deps',  # try without deps first
        'vllm'
    ])
    print(f"pip install (no-deps) exit code: {result.returncode}")
    
    if result.returncode != 0:
        # Try installing the wheel directly
        if vllm_wheels:
            whl_path = os.path.join(wheels_dir, vllm_wheels[0])
            print(f"Trying direct install: {whl_path}")
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', whl_path])
    
    # Now install remaining deps
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--no-index', '--find-links', wheels_dir,
        'vllm'
    ])
else:
    print(f"Archive not found at {archive}")
    # Search everywhere
    archives = glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
    print(f"All archives found: {archives}")

try:
    import vllm
    print(f"\nSUCCESS: vLLM {vllm.__version__}")
except ImportError as e:
    print(f"\nFAILED: {e}")
    # Last resort: check if vllm is available some other way
    result = subprocess.run([sys.executable, '-c', 'import vllm; print(vllm.__version__)'], 
                          capture_output=True, text=True)
    print(f"Direct check: stdout={result.stdout.strip()}, stderr={result.stderr.strip()}")

In [ ]:
# ============================================================
# Cell 0: Install vLLM from bundled wheels (no internet)
# Source: andreasbis/aimo-3-utils kernel output
# ============================================================
import os, sys, subprocess

def install_deps():
    """Install vLLM and deps from pre-packaged wheels."""
    archive = '/kaggle/input/aimo-3-utils/wheels.tar.gz'
    temp_dir = '/kaggle/tmp/setup'
    
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', 
        '--no-index', '--find-links', f'{temp_dir}/wheels',
        'vllm'
    ], check=True, capture_output=True)
    print("vLLM installed successfully.")

install_deps()

In [ ]:
# ============================================================
# Cell 1: Configuration & Imports
# ============================================================
import os, sys, time, re, json, traceback, threading, gc
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr
from collections import Counter
from pathlib import Path

import polars as pl

# === CONFIGURATION ===
MODEL_PATH = "/kaggle/input/gpt-oss-120b/transformers/default/1"
N_SAMPLES = 32                # Solution attempts per problem
MAX_TOKENS = 8192             # Max tokens per generation
TEMPERATURE = 0.7             # Sampling temperature
TOP_P = 0.95
CODE_TIMEOUT = 30             # Seconds per code execution
TIR_MAX_RETRIES = 3           # Max code-execute-continue cycles per solution
ANSWER_MOD = 100_000
GPU_MEMORY_UTIL = 0.92
MAX_MODEL_LEN = 16384
KV_CACHE_DTYPE = "fp8_e4m3"   # FP8 KV cache for 120B model on H100

print(f"Config: model={MODEL_PATH}")
print(f"  N={N_SAMPLES}, tokens={MAX_TOKENS}, temp={TEMPERATURE}, kv_cache={KV_CACHE_DTYPE}")

In [ ]:
# ============================================================
# Cell 2: Lazy Model Loader
# Model loads on first predict() call, NOT before serve().
# serve() MUST be called within 15 minutes of script start.
# ============================================================
from vllm import LLM, SamplingParams

class LazyModel:
    """Lazy-loading wrapper for vLLM. Loads on first use."""
    def __init__(self):
        self._llm = None
        self._tokenizer = None
    
    def _load(self):
        if self._llm is not None:
            return
        print(f"Loading model from {MODEL_PATH}...")
        t0 = time.time()
        self._llm = LLM(
            model=MODEL_PATH,
            tensor_parallel_size=1,
            gpu_memory_utilization=GPU_MEMORY_UTIL,
            max_model_len=MAX_MODEL_LEN,
            dtype="auto",
            kv_cache_dtype=KV_CACHE_DTYPE,
            enable_prefix_caching=True,
            max_num_seqs=64,
            trust_remote_code=True,
        )
        self._tokenizer = self._llm.get_tokenizer()
        print(f"Model loaded in {time.time() - t0:.1f}s")
    
    @property
    def llm(self):
        self._load()
        return self._llm
    
    @property
    def tokenizer(self):
        self._load()
        return self._tokenizer

model = LazyModel()
print("Lazy model wrapper ready (loads on first predict call).")

In [ ]:
# ============================================================
# Cell 3: Answer Extraction
# ============================================================

def _parse_number(s: str) -> int | None:
    """Parse a string to integer."""
    s = s.strip().replace("\\,", "").replace("\\;", "").replace("\\!", "")
    s = s.replace(",", "")
    s = re.sub(r"\\text\{.*?\}", "", s)
    s = re.sub(r"\\mathrm\{.*?\}", "", s)
    try:
        return int(s)
    except ValueError:
        pass
    try:
        f = float(s)
        if f == int(f) and not (f != f):
            return int(f)
    except (ValueError, OverflowError):
        pass
    match = re.search(r"(-?\d+(?:\.\d+)?)", s)
    if match:
        try:
            f = float(match.group(1))
            if f == int(f):
                return int(f)
        except (ValueError, OverflowError):
            pass
    return None


def extract_boxed(text: str) -> int | None:
    """Extract answer from \\boxed{...} with nested brace handling."""
    idx = text.rfind("\\boxed")
    if idx == -1:
        return None
    brace_start = text.find("{", idx)
    if brace_start == -1:
        return None
    depth = 0
    end = brace_start
    for i in range(brace_start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                end = i
                break
    content = text[brace_start + 1:end].strip()
    return _parse_number(content)


def extract_from_code_output(text: str) -> int | None:
    """Extract from ```output ... ``` blocks."""
    matches = re.findall(r"```output\s*(.*?)```", text, re.DOTALL)
    if not matches:
        return None
    return _parse_number(matches[-1].strip())


def extract_natural_language(text: str) -> int | None:
    """Extract from 'the answer is N' patterns."""
    for pattern in [
        r"(?:the\s+)?(?:final\s+)?answer\s+is\s*[:\s]*(-?\d+(?:\.\d+)?)",
        r"(?:therefore|thus|hence|so)\s*,?\s*(?:the\s+answer\s+is\s+)?(-?\d+(?:\.\d+)?)",
        r"answer\s*[=:]\s*(-?\d+(?:\.\d+)?)",
    ]:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return _parse_number(matches[-1])
    return None


def extract_last_integer(text: str) -> int | None:
    """Fallback: last integer in text."""
    matches = re.findall(r"(?<![.\d])(-?\d+)(?![.\d])", text)
    if matches:
        return _parse_number(matches[-1])
    return None


def extract_answer(text: str) -> int | None:
    """Extract integer answer using multiple strategies."""
    for fn in [extract_boxed, extract_from_code_output, extract_natural_language, extract_last_integer]:
        result = fn(text)
        if result is not None:
            return result % ANSWER_MOD
    return None

print("Answer extraction ready.")

In [ ]:
# ============================================================
# Cell 4: Code Execution Sandbox
# ============================================================

SANDBOX_IMPORTS = """
import math
import numpy as np
import sympy as sp
from sympy import *
from sympy import symbols, solve, simplify, expand, factor, Rational, sqrt, oo
from sympy import pi, E, I, sin, cos, tan, log, exp, Abs, floor, ceiling
from sympy import gcd, lcm, isprime, nextprime, factorint, divisors, totient
from sympy import binomial, factorial, fibonacci
from sympy import Matrix, det, eye
from sympy.ntheory import mobius, primerange
from itertools import combinations, permutations, product as iproduct
from collections import Counter, defaultdict
from fractions import Fraction
from functools import reduce
import itertools
"""

def execute_code(code: str, timeout: int = CODE_TIMEOUT):
    """Execute Python code with timeout. Returns (stdout, stderr, success)."""
    stdout_buf = StringIO()
    stderr_buf = StringIO()
    namespace = {}
    try:
        exec(SANDBOX_IMPORTS, namespace)
    except ImportError:
        pass
    
    exception_holder = [None]
    def _run():
        try:
            with redirect_stdout(stdout_buf), redirect_stderr(stderr_buf):
                exec(code, namespace)
        except Exception as e:
            exception_holder[0] = e
    
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    thread.join(timeout=timeout)
    
    if thread.is_alive():
        return "", f"Timeout after {timeout}s", False
    
    stdout = stdout_buf.getvalue()
    stderr = stderr_buf.getvalue()
    
    if exception_holder[0] is not None:
        exc = exception_holder[0]
        tb = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
        return stdout, tb, False
    
    return stdout, stderr, True


def extract_code_blocks(text: str) -> list[str]:
    """Extract Python code blocks from text."""
    blocks = re.findall(r"```python\s*\n(.*?)```", text, re.DOTALL)
    if blocks:
        return blocks
    return re.findall(r"```\s*\n(.*?)```", text, re.DOTALL)

print("Code sandbox ready.")

In [ ]:
# ============================================================
# Cell 5: Problem Classifier
# ============================================================

GEO_KW = [r"\\triangle", r"\\angle", r"circle", r"polygon", r"quadrilateral",
          r"diameter", r"radius", r"tangent", r"perpendicular", r"bisect",
          r"midpoint", r"\\overline", r"area of", r"perimeter", r"right triangle",
          r"isosceles", r"equilateral", r"\\sin", r"\\cos", r"coordinate"]

NT_KW = [r"\\bmod", r"\\pmod", r"modulo", r"remainder when", r"divisible",
         r"divisor", r"\\gcd", r"greatest common", r"prime", r"coprime",
         r"congruent", r"residue", r"perfect square", r"digit sum", r"factorial"]

COMBO_KW = [r"how many", r"number of ways", r"probability", r"expected value",
            r"permutation", r"combination", r"\\binom", r"arrange", r"distribute",
            r"subset", r"coloring", r"distinct", r"grid", r"partition"]

ALG_KW = [r"polynomial", r"equation", r"inequality", r"\\sum", r"\\prod",
          r"maximum", r"minimum", r"function", r"real number", r"root",
          r"coefficient", r"matrix", r"determinant", r"\\lfloor", r"\\rfloor"]

def classify_problem(problem: str) -> str:
    p = problem.lower()
    scores = {}
    for name, kws in [("geometry", GEO_KW), ("number_theory", NT_KW),
                       ("combinatorics", COMBO_KW), ("algebra", ALG_KW)]:
        scores[name] = sum(1 for kw in kws if re.search(kw, p, re.IGNORECASE))
    best = max(scores, key=scores.get)
    return best if scores[best] >= 1 else "default"

print("Problem classifier ready.")

In [ ]:
# ============================================================
# Cell 6: Prompt Templates
# ============================================================

TIR_SYSTEM_PROMPT = """You are a world-class mathematician. Solve the given problem step by step.

IMPORTANT INSTRUCTIONS:
- Write Python code to help solve the problem. Use ```python ... ``` blocks.
- After code execution, you will see results in ```output ... ``` blocks.
- Use sympy for symbolic computation, numpy for numerical work.
- You may write multiple code blocks, each building on previous results.
- After reaching the answer, put it inside \\boxed{N} where N is an integer.
- If the problem says "find the remainder when X is divided by Y", compute X % Y.
- Double-check your answer with a verification code block before giving \\boxed{}.
- The final answer must be a non-negative integer between 0 and 99999."""

TYPE_INSTRUCTIONS = {
    "algebra": "This is an algebra problem. Use sympy.symbols() and sympy.solve(). Verify by substitution.",
    "combinatorics": "This is a combinatorics problem. Enumerate small cases first. Use itertools and sympy.binomial().",
    "geometry": "This is a geometry problem. Set up coordinates. Use sympy for symbolic computation. Verify numerically.",
    "number_theory": "This is a number theory problem. Use sympy.factorint(), pow(b,e,m), sympy.gcd(). Check small cases.",
    "default": "Solve step by step. Use Python code to compute and verify.",
}


def build_prompt(problem: str, problem_type: str = "default") -> str:
    """Build a chat-template prompt for the model."""
    type_hint = TYPE_INSTRUCTIONS.get(problem_type, TYPE_INSTRUCTIONS["default"])
    messages = [
        {"role": "system", "content": TIR_SYSTEM_PROMPT},
        {"role": "user", "content": f"{type_hint}\n\nProblem:\n{problem}"},
    ]
    return model.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Prompt templates ready.")

In [ ]:
# ============================================================
# Cell 7: TIR Batch Solver
# ============================================================

def tir_solve_batch(prompt: str, n_samples: int = N_SAMPLES):
    """Generate N solutions with TIR, execute code, extract answers.
    
    Returns list of (answer, code_executed, code_succeeded, has_boxed) tuples.
    """
    # Step 1: Generate N initial completions (batched via vLLM)
    params = SamplingParams(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_TOKENS,
        n=n_samples,
        stop=["```output"],
    )
    outputs = model.llm.generate([prompt], params, use_tqdm=False)
    completions = [o.text for o in outputs[0].outputs]
    
    results = []
    
    for comp in completions:
        full_text = comp
        code_executed = False
        code_succeeded = False
        
        # Check for code blocks
        code_blocks = extract_code_blocks(comp)
        
        if code_blocks:
            code = code_blocks[-1]
            stdout, stderr, success = execute_code(code)
            code_executed = True
            code_succeeded = success
            
            if success:
                output_text = stdout.strip() if stdout.strip() else "(no output)"
            else:
                output_text = stderr.strip()[:500]
            
            # Append output and continue generation
            full_text += f"\n```output\n{output_text}\n```\n"
            
            # Continue generation (greedy, single completion)
            cont_prompt = prompt + full_text
            for retry in range(TIR_MAX_RETRIES):
                cont_params = SamplingParams(
                    temperature=0.0,
                    max_tokens=MAX_TOKENS // 2,
                    n=1,
                    stop=["```output"],
                )
                cont_out = model.llm.generate([cont_prompt], cont_params, use_tqdm=False)
                chunk = cont_out[0].outputs[0].text if cont_out[0].outputs else ""
                if not chunk.strip():
                    break
                    
                full_text += chunk
                
                # Check for more code
                new_blocks = extract_code_blocks(chunk)
                if new_blocks:
                    stdout, stderr, success = execute_code(new_blocks[-1])
                    if success:
                        output_text = stdout.strip() if stdout.strip() else "(no output)"
                        code_succeeded = True
                    else:
                        output_text = stderr.strip()[:500]
                    full_text += f"\n```output\n{output_text}\n```\n"
                    cont_prompt = prompt + full_text
                else:
                    break
                
                if extract_boxed(full_text) is not None:
                    break
        
        answer = extract_answer(full_text)
        has_boxed = "\\boxed" in full_text
        results.append((answer, code_executed, code_succeeded, has_boxed))
    
    return results

print("TIR solver ready.")

In [ ]:
# ============================================================
# Cell 8: Voting
# ============================================================

def vote(results: list[tuple]) -> tuple[int, float]:
    """Weighted majority vote. Returns (answer, confidence)."""
    answers = []
    weights = []
    
    for answer, code_exec, code_ok, has_boxed in results:
        if answer is None:
            continue
        w = 1.0
        if code_exec and code_ok:
            w += 2.0
        elif code_exec:
            w += 0.5
        if has_boxed:
            w += 0.5
        answers.append(answer)
        weights.append(w)
    
    if not answers:
        return 0, 0.0
    
    # Weighted vote
    weighted = {}
    for a, w in zip(answers, weights):
        weighted[a] = weighted.get(a, 0.0) + w
    
    best = max(weighted, key=weighted.get)
    
    # Confidence = fraction agreeing with majority
    counter = Counter(answers)
    confidence = counter.most_common(1)[0][1] / len(answers)
    
    return best % ANSWER_MOD, confidence

print("Voting ready.")

In [ ]:
# ============================================================
# Cell 9: Time Manager + Problem Counter
# ============================================================

class TimeManager:
    def __init__(self, total_limit=32400, per_limit=1700, n_problems=110):
        self.total_limit = total_limit
        self.per_limit = per_limit
        self.n_problems = n_problems
        self.start = time.time()
        self.solved = 0
        self.times = []
    
    def elapsed(self): return time.time() - self.start
    def remaining(self): return max(0, self.total_limit - self.elapsed())
    
    def budget(self):
        rem = max(1, self.n_problems - self.solved)
        return max(60.0, min(self.remaining() / rem, self.per_limit))
    
    def get_n_samples(self, base=N_SAMPLES):
        b = self.budget()
        if b >= 1500: return base
        elif b >= 900: return max(16, base // 2)
        elif b >= 300: return max(8, base // 4)
        else: return 4
    
    def record(self, t):
        self.solved += 1
        self.times.append(t)
    
    def should_skip(self): return self.remaining() < 30
    
    def status(self):
        avg = sum(self.times)/len(self.times) if self.times else 0
        return f"{self.solved}/{self.n_problems} | {self.elapsed():.0f}s elapsed | {self.remaining():.0f}s left | avg {avg:.1f}s"

timer = TimeManager()
print("Time manager ready.")

In [ ]:
# ============================================================
# Cell 10: Main Solve Function
# ============================================================

def solve(problem: str) -> int:
    """Solve a single AIMO problem. Returns integer answer 0-99999."""
    if timer.should_skip():
        print("  WARNING: Time critical, returning 0")
        return 0
    
    t0 = time.time()
    
    try:
        # Classify
        ptype = classify_problem(problem)
        
        # Build prompt
        prompt = build_prompt(problem, ptype)
        
        # Adaptive N
        n = timer.get_n_samples()
        
        # TIR solve
        results = tir_solve_batch(prompt, n_samples=n)
        
        # Vote
        answer, confidence = vote(results)
        
        valid = sum(1 for r in results if r[0] is not None)
        code_ok = sum(1 for r in results if r[2])
        
        elapsed = time.time() - t0
        timer.record(elapsed)
        
        print(f"  type={ptype} | n={n} | valid={valid} | code_ok={code_ok} | "
              f"conf={confidence:.2f} | answer={answer} | {elapsed:.1f}s | {timer.status()}")
        
        return answer
    
    except Exception as e:
        elapsed = time.time() - t0
        timer.record(elapsed)
        print(f"  ERROR: {e} | returning 0 | {elapsed:.1f}s")
        traceback.print_exc()
        return 0

print("Solver ready.")

In [ ]:
# ============================================================
# Cell 11: Kaggle Submission Server
# ============================================================
# API: kaggle_evaluation.aimo_3_inference_server
# - predict() is called once per problem via gRPC
# - Must call serve() within 15 minutes of notebook start
# - Returns pl.DataFrame({'id': str, 'answer': int})
# ============================================================

import kaggle_evaluation.aimo_3_inference_server

def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame:
    """Prediction function called by the Kaggle gRPC gateway.
    
    Args:
        id_: Polars Series with one problem ID string
        problem: Polars Series with one LaTeX problem string
    
    Returns:
        Polars DataFrame with columns 'id' (str) and 'answer' (int 0-99999)
    """
    problem_id = id_.item(0)
    problem_text = problem.item(0)
    
    print(f"\n{'='*60}")
    print(f"Problem {timer.solved + 1} (id={problem_id}):")
    print(f"  {problem_text[:120]}..." if len(problem_text) > 120 else f"  {problem_text}")
    
    answer = solve(problem_text)
    
    # Ensure answer is valid integer in range
    answer = int(answer) % ANSWER_MOD
    
    print(f"  => Final answer: {answer}")
    
    return pl.DataFrame({'id': [problem_id], 'answer': [answer]})


# Create and start the inference server
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Competition evaluation mode
    inference_server.serve()
else:
    # Local testing mode (uses test.csv from competition data)
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )

print(f"\nDone! {timer.status()}")